# Handwritten Digits Classification Using KNN

This notebook follows the original project workflow:

1. Load MNIST
2. Explore image shape and labels
3. Visualize handwritten digits
4. Normalize pixel values
5. Flatten 28×28 images into 784 features
6. Train KNN with `n_neighbors=3`
7. Evaluate accuracy
8. Test individual predictions
9. Rotate an MNIST image using OpenCV
10. Process an external handwritten image
11. Predict the external digit
12. Save the trained KNN model with Joblib

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import joblib

from tensorflow.keras import datasets
from sklearn.neighbors import KNeighborsClassifier

print("Python executable:", sys.executable)

In [ ]:
# Load MNIST
(X_train, y_train), (X_test, y_test) = datasets.mnist.load_data()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("X_train type:", type(X_train))

In [ ]:
# Inspect the first training image and label
print("First image shape:", X_train[0].shape)
print("First label:", y_train[0])

plt.figure(figsize=(5, 5))
plt.imshow(X_train[0], cmap="gray")
plt.title(f"Training Digit: {y_train[0]}")
plt.axis("off")
plt.show()

In [ ]:
# Heatmap visualization
plt.figure(figsize=(6, 5))
sns.heatmap(X_train[0], cmap="gray", cbar=True)
plt.title("MNIST Digit Pixel Heatmap")
plt.show()

In [ ]:
# Normalize pixel values: 0-255 -> 0-1
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

print(X_train[0])

In [ ]:
# Flatten 28x28 images into 784 features
X_train_flattened = X_train.reshape(len(X_train), 28 * 28)
X_test_flattened = X_test.reshape(len(X_test), 28 * 28)

print("X_train_flattened shape:", X_train_flattened.shape)
print("X_test_flattened shape:", X_test_flattened.shape)

In [ ]:
# Verify that flattening can be reversed
plt.figure(figsize=(5, 5))
plt.imshow(X_train_flattened[0].reshape(28, 28), cmap="gray")
plt.title(f"Flattened and Reshaped Digit: {y_train[0]}")
plt.axis("off")
plt.show()

## Train the KNN classifier

The original project uses `KNeighborsClassifier(n_neighbors=3)`.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_flattened, y_train)

accuracy = knn.score(X_test_flattened, y_test)
print(f"KNN test accuracy: {accuracy:.4f}")

In [ ]:
# Test one sample
sample_index = 0

predicted = knn.predict(X_test_flattened[[sample_index]])[0]
actual = y_test[sample_index]

print("Predicted:", predicted)
print("Actual:", actual)

plt.figure(figsize=(5, 5))
plt.imshow(X_test[sample_index], cmap="gray")
plt.title(f"Predicted: {predicted} | Actual: {actual}")
plt.axis("off")
plt.show()

In [ ]:
# Predict the complete test set
prediction = knn.predict(X_test_flattened)

print("First prediction:", prediction[0])
print("First actual label:", y_test[0])

In [ ]:
# Inspect one flattened test image with Pandas
sample_index = 7

df = pd.DataFrame(X_test_flattened[sample_index]).T
display(df)

## Rotation experiment

The original notebook rotates one MNIST test image by 90 degrees and checks the KNN prediction.

In [ ]:
sample_index = 6

image = X_test[sample_index]
h, w = image.shape[:2]

center = (w / 2, h / 2)
matrix = cv2.getRotationMatrix2D(center, 90, 1)
rotimg = cv2.warpAffine(image, matrix, (w, h))

print("Rotated image shape:", rotimg.shape)

plt.figure(figsize=(5, 5))
plt.imshow(rotimg, cmap="gray")
plt.title("Rotated MNIST Image")
plt.axis("off")
plt.show()

In [ ]:
# Flatten the rotated image and predict
rotimg_flat = rotimg.reshape(1, 28 * 28)

rotated_prediction = knn.predict(rotimg_flat)
print("Prediction for rotated image:", rotated_prediction)

## External handwritten image

Place an external handwritten digit image at:

`images/modified.png`

The image is converted to grayscale, inverted, resized to 28×28, normalized, flattened, and sent to KNN.

In [ ]:
external_path = Path("images/modified.png")

if not external_path.exists():
    print(f"Place your image at: {external_path}")
else:
    modified_image = cv2.imread(str(external_path), cv2.IMREAD_GRAYSCALE)

    if modified_image is None:
        raise ValueError("The image could not be read.")

    print("Original external image shape:", modified_image.shape)

    plt.figure(figsize=(6, 6))
    plt.imshow(modified_image, cmap="gray")
    plt.title("Original External Image")
    plt.axis("off")
    plt.show()

In [ ]:
if external_path.exists():
    # Invert white background / black digit into MNIST-like black background / white digit
    img_resizedM = cv2.bitwise_not(modified_image)

    # Resize to MNIST dimensions
    img_resizedM = cv2.resize(
        img_resizedM,
        (28, 28),
        interpolation=cv2.INTER_LINEAR
    )

    print("Resized image shape:", img_resizedM.shape)

    plt.figure(figsize=(5, 5))
    plt.imshow(img_resizedM, cmap="gray")
    plt.title("External Image Resized to 28×28")
    plt.axis("off")
    plt.show()

In [ ]:
if external_path.exists():
    # Normalize and flatten
    img_resizedM = img_resizedM.astype(np.float32) / 255.0
    img_resizedM_Flat = img_resizedM.reshape(1, 28 * 28)

    print("External flattened shape:", img_resizedM_Flat.shape)

    external_prediction = knn.predict(img_resizedM_Flat)
    print("Predicted external digit:", external_prediction[0])

## Save the trained model

In [ ]:
Path("models").mkdir(exist_ok=True)

model_path = "models/KNNModel_Job"
joblib.dump(knn, model_path)

print(f"Model saved: {model_path}")

In [ ]:
# Optional: load the saved model and predict again
loaded_knn = joblib.load("models/KNNModel_Job")

if external_path.exists():
    print("Loaded model prediction:", loaded_knn.predict(img_resizedM_Flat))